In [44]:
import torch
import librosa
import numpy as np
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import parselmouth
from parselmouth.praat import call
import re
import spacy
import language_tool_python
from collections import Counter

In [45]:
nlp = spacy.load("en_core_web_sm")

grammar_tool = language_tool_python.LanguageTool("en-US")

In [46]:
file_dir = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/"

In [47]:
processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")   

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

In [48]:
def transcribe(file_path):
    audio, _ = librosa.load(file_path, sr=16000)

    input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features

    with torch.no_grad():
        output_ids = model.generate(input_features)

    return processor.batch_decode(output_ids, skip_special_tokens=True)[0]

In [49]:
def extract_features(file_path, transcription=None):

    audio, sr = librosa.load(file_path, sr=16000)

    features = {}

    # --------------------------------------------------
    # Speech / Silence Detection
    # --------------------------------------------------

    speech_segments = librosa.effects.split(
        audio,
        top_db=30
    )

    total_duration = len(audio) / sr

    speech_duration = sum(
        (end - start) / sr
        for start, end in speech_segments
    )

    features["speech_duration"] = float(speech_duration)

    # Proportion of audio containing speech
    features["speech_ratio"] = (
        float(speech_duration / total_duration)
        if total_duration > 0
        else 0.0
    )

    # --------------------------------------------------
    # Pauses
    # --------------------------------------------------

    pauses = []

    for i in range(len(speech_segments) - 1):

        gap = (
            speech_segments[i + 1][0]
            - speech_segments[i][1]
        ) / sr

        # Ignore extremely small gaps
        if gap > 0.1:
            pauses.append(gap)

    features["num_pauses"] = int(len(pauses))

    features["pause_mean"] = (
        float(np.mean(pauses))
        if pauses
        else 0.0
    )

    features["pause_max"] = (
        float(np.max(pauses))
        if pauses
        else 0.0
    )

    # --------------------------------------------------
    # Total Silence
    # --------------------------------------------------

    features["silence_duration"] = float(
        total_duration - speech_duration
    )

    # --------------------------------------------------
    # Speaking Rate
    # --------------------------------------------------

    if transcription and speech_duration > 0:

        words = transcription.split()

        features["speaking_rate"] = float(
            len(words) / speech_duration
        )

    else:

        features["speaking_rate"] = 0.0

    return features

In [50]:
import re
import numpy as np
from collections import Counter


def extract_text_features(transcription):

    features = {}

    # =========================================================
    # CLEAN TEXT
    # =========================================================

    text = str(transcription).strip()
    text = re.sub(r"\s+", " ", text)

    doc = nlp(text)

    words = [
        token for token in doc
        if token.is_alpha
    ]

    n_words = len(words)

    sentences = list(doc.sents)
    n_sentences = len(sentences)

    # =========================================================
    # SENTENCE STRUCTURE
    # =========================================================

    features["word_count"] = n_words
    features["sentence_count"] = n_sentences

    sentence_lengths = [
        sum(1 for token in sent if token.is_alpha)
        for sent in sentences
    ]

    if sentence_lengths:

        features["avg_sentence_length"] = float(
            np.mean(sentence_lengths)
        )

        features["std_sentence_length"] = float(
            np.std(sentence_lengths)
        )

    else:

        features["avg_sentence_length"] = 0.0
        features["std_sentence_length"] = 0.0

    # =========================================================
    # LEXICAL DIVERSITY
    # =========================================================

    unique_words = set(
        token.lemma_.lower()
        for token in words
    )

    features["unique_word_ratio"] = (
        len(unique_words) / n_words
        if n_words > 0 else 0.0
    )

    # =========================================================
    # VERB / TENSE FEATURES
    # =========================================================

    verbs = [
        token for token in words
        if token.pos_ in {"VERB", "AUX"}
    ]

    features["verb_ratio"] = (
        len(verbs) / n_words
        if n_words > 0 else 0.0
    )

    tense_counts = Counter()

    for token in verbs:

        tense = token.morph.get("Tense")

        if tense:
            tense_counts[tense[0]] += 1

    total_tensed_verbs = sum(
        tense_counts.values()
    )

    features["past_tense_ratio"] = (
        tense_counts["Past"] / total_tensed_verbs
        if total_tensed_verbs > 0 else 0.0
    )

    features["present_tense_ratio"] = (
        tense_counts["Pres"] / total_tensed_verbs
        if total_tensed_verbs > 0 else 0.0
    )

    features["tense_variety"] = len(
        tense_counts
    )

    # =========================================================
    # SYNTAX COMPLEXITY
    # =========================================================

    dependency_depths = [
        len(list(token.ancestors))
        for token in words
    ]

    if dependency_depths:

        features["avg_dependency_depth"] = float(
            np.mean(dependency_depths)
        )

        features["max_dependency_depth"] = float(
            np.max(dependency_depths)
        )

    else:

        features["avg_dependency_depth"] = 0.0
        features["max_dependency_depth"] = 0.0

    # =========================================================
    # CLAUSE COMPLEXITY
    # =========================================================

    subordinate_deps = {
        "advcl",
        "ccomp",
        "xcomp",
        "acl",
        "relcl"
    }

    features["subordinate_clause_count"] = sum(
        token.dep_ in subordinate_deps
        for token in doc
    )

    features["relative_clause_count"] = sum(
        token.dep_ == "relcl"
        for token in doc
    )

    features["coordination_count"] = sum(
        token.dep_ == "conj"
        for token in doc
    )

    # =========================================================
    # INCOMPLETE / FRAGMENTED SENTENCES
    # =========================================================

    fragment_count = 0

    for sent in sentences:

        sent_tokens = [
            token for token in sent
            if token.is_alpha
        ]

        has_verb = any(
            token.pos_ in {"VERB", "AUX"}
            for token in sent_tokens
        )

        if len(sent_tokens) >= 3 and not has_verb:
            fragment_count += 1

    features["fragment_ratio"] = (
        fragment_count / n_sentences
        if n_sentences > 0 else 0.0
    )

    # =========================================================
    # GRAMMAR ERRORS
    # =========================================================

    try:

        matches = grammar_tool.check(text)

        grammar_errors = []

        for match in matches:

            category = str(
                getattr(match, "category", "")
            ).upper()

            # Ignore spelling/capitalization-only issues
            if (
                "TYPOS" not in category
                and "SPELL" not in category
                and "CASING" not in category
            ):
                grammar_errors.append(match)

        features["grammar_error_count"] = len(
            grammar_errors
        )

        features["grammar_error_rate"] = (
            len(grammar_errors) / n_words
            if n_words > 0 else 0.0
        )

    except Exception:

        features["grammar_error_count"] = 0.0
        features["grammar_error_rate"] = 0.0

    return features

In [51]:
def extract_features(file_path, transcription=None):

    audio, sr = librosa.load(
        file_path,
        sr=16000
    )

    features = {}

    # =========================================================
    # SPEECH / SILENCE SEGMENTS
    # =========================================================

    speech_segments = librosa.effects.split(
        audio,
        top_db=30
    )

    total_duration = len(audio) / sr

    speech_duration = sum(
        (end - start) / sr
        for start, end in speech_segments
    )

    features["speech_duration"] = float(
        speech_duration
    )

    features["speech_ratio"] = (
        float(speech_duration / total_duration)
        if total_duration > 0
        else 0.0
    )

    # =========================================================
    # PAUSES
    # =========================================================

    pauses = []

    for i in range(len(speech_segments) - 1):

        gap = (
            speech_segments[i + 1][0]
            - speech_segments[i][1]
        ) / sr

        # Gaps > 100 ms treated as pauses
        if gap > 0.1:
            pauses.append(gap)

    features["num_pauses"] = int(
        len(pauses)
    )

    features["pause_mean"] = (
        float(np.mean(pauses))
        if pauses
        else 0.0
    )

    features["pause_max"] = (
        float(np.max(pauses))
        if pauses
        else 0.0
    )

    # =========================================================
    # SILENCE
    # =========================================================

    features["silence_duration"] = float(
        total_duration - speech_duration
    )

    # =========================================================
    # SPEAKING RATE
    # =========================================================

    if transcription and speech_duration > 0:

        word_count = len(
            transcription.split()
        )

        features["speaking_rate"] = float(
            word_count / speech_duration
        )

    else:

        features["speaking_rate"] = 0.0

    return features

In [ ]:
print(process_audio(f"{file_dir}/train/audio_0.wav"))